# 🎬 PeakCut AI Video Clipper on Google Colab (Free T4 GPU + 5 TB Google Drive)

Run PeakCut entirely in Google Cloud with free T4 GPU acceleration, ultra-fast 100 MB/s network downloads, persistent 5 TB Google Drive storage, and access the web dashboard from your browser via a secure Cloudflare Tunnel.

**Enable GPU:** Click `Runtime` -> `Change runtime type` -> select `T4 GPU`.

In [ ]:
# 1. Mount 5 TB Google Drive (Permanent Storage for Database, Videos, and Clips)
from google.colab import drive
import os

drive.mount('/content/drive')
os.environ["PEAKCUT_DATA"] = "/content/drive/MyDrive/PeakCut_Data"
os.environ["PEAKCUT_ALLOW_TUNNELS"] = "1"
os.makedirs("/content/drive/MyDrive/PeakCut_Data", exist_ok=True)
print("✅ 5 TB Google Drive mounted successfully! All videos and accounts will be saved permanently.")

In [ ]:
# 2. Clone repository and install dependencies
!git clone https://github.com/meet1785/pekacut-clipper.git /content/PeakCut
%cd /content/PeakCut
!git pull
!pip install -q -r requirements.txt
!apt-get install -y -qq ffmpeg

# Download Cloudflare tunnel binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

In [ ]:
# 3. Start PeakCut Studio Server and Public Cloudflare Tunnel
import subprocess, time, re, os

os.environ["PEAKCUT_ALLOW_TUNNELS"] = "1"

# Kill any old instance if re-running cell
!pkill -f "studio.server" || true

# Start PeakCut server in background
server_proc = subprocess.Popen(["python", "-m", "studio.server"], cwd="/content/PeakCut", env=os.environ)
time.sleep(3)

# Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8765"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Waiting for Cloudflare Tunnel URL...")
public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("\n" + "="*60)
    print(f"🚀 PEAKCUT IS LIVE IN THE CLOUD!")
    print(f"👉 Open your dashboard: {public_url}")
    print("="*60 + "\n")
else:
    print("Tunnel did not output URL in time.")
